# 🚀 Entrenamiento de YOLOv8 conectando Google Drive

Este notebook monta tu **Google Drive**, descomprime automáticamente `data.zip` desde tu unidad y guarda el modelo final **`best.pt`** en Google Drive.

In [ ]:
# 1. Verificar aceleración por GPU
!nvidia-smi

In [ ]:
# 2. Instalar Ultralytics YOLOv8
!pip install -q ultralytics

In [ ]:
# 3. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. Descomprimir data.zip desde Google Drive (MyDrive)
import zipfile
import os

zip_path = '/content/drive/MyDrive/data.zip'

if os.path.exists(zip_path):
    print('📦 Descomprimiendo data.zip desde Google Drive...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content')
    print('✅ Dataset listo en /content/data')
else:
    # Búsqueda por si está en una subcarpeta de Google Drive
    found = False
    print('🔍 Buscando data.zip en Google Drive...')
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if 'data.zip' in files:
            target_file = os.path.join(root, 'data.zip')
            print(f'📦 Encontrado en {target_file}. Descomprimiendo...')
            with zipfile.ZipFile(target_file, 'r') as zip_ref:
                zip_ref.extractall('/content')
            found = True
            print('✅ Dataset listo en /content/data')
            break
    if not found:
        print('❌ No se encontró data.zip en Google Drive. Verifica que esté subido.')

In [ ]:
# 5. Crear archivo data.yaml
import yaml
from pathlib import Path

data_yaml = {
    'train': str(Path('/content/data/train/images').resolve()),
    'val': str(Path('/content/data/val/images').resolve()),
    'nc': 1,
    'names': ['plate']
}

with open('data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print('✅ Archivo data.yaml configurado.')

In [ ]:
# 6. Entrenar el modelo YOLOv8 en GPU
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data='data.yaml',
    epochs=80,
    imgsz=640,
    batch=16,
    device=0,  # GPU
    patience=20,
    project='runs',
    name='retrain_plate_v1',
    seed=42,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15
)
print('🏆 ¡Entrenamiento finalizado!')

In [ ]:
# 7. Guardar una copia directa de best.pt en tu Google Drive
import shutil
from google.colab import files

best_weights = 'runs/retrain_plate_v1/weights/best.pt'
drive_output = '/content/drive/MyDrive/best.pt'

if os.path.exists(best_weights):
    shutil.copy2(best_weights, drive_output)
    print(f'✅ ¡Modelo guardado en tu Google Drive!: {drive_output}')
    files.download(best_weights)
else:
    print('No se encontró el archivo de pesos.')